<a href="https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zafar488/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

## My Provisional Lane

I have selected **Lane 2: Refresh / Content Opportunity Scoring**.

I chose this lane because a search or content team may have thousands of pages but limited time to review them. The practical problem is therefore not simply identifying pages with weak performance, but deciding which pages should be reviewed first.

This lane will allow me to investigate whether observable signals such as impressions, average position, CTR, content age, freshness, engagement, and word count can support a transparent ranked review queue.

My intended output is not an automatic decision to edit a page. It is a decision-support system that helps a human reviewer prioritize pages for refresh, expansion, CTR improvement, protection, or monitoring.

This lane is provisional and may be refined before the end of Week 4 as I learn more about the warehouse data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

## Research Question

**Among pages with enough search visibility, which pages should a content or SEO team review first for a possible refresh or other content action?**

### Unit of Analysis

The unit of analysis is **one content page**.

Each row represents one anonymized page with observable search, content, and engagement signals.

### Decision

The decision being improved is:

> Which page should the content team review next?

### Expected Output

The intended output is a ranked review queue containing:

- a page identifier;
- an opportunity or review score;
- a suggested action;
- one or more reason codes;
- a confidence level.

Possible actions may include:

- refresh the content;
- expand or improve the page;
- review title and metadata;
- protect a currently strong page;
- monitor the page;
- take no immediate action.

### Human Action

A content or SEO reviewer would inspect the highest-ranked pages and decide whether an update is justified.

The model or scoring system would support the reviewer, not replace human judgment.

### Cost of a Wrong Recommendation

A false positive could cause the team to spend time reviewing or changing a page that did not need attention. An unnecessary change could also affect a page that was already performing well.

A false negative could cause the team to miss a page with a meaningful decline or opportunity.

Because review capacity is limited, the quality of the top-ranked recommendations is more important than generic accuracy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
from pathlib import Path
import subprocess
import numpy as np
import pandas as pd

# Starter dataset ko locate karo
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")
]

data_path = next((path for path in possible_paths if path.exists()), None)

# Agar dataset available nahi hai to public starter repo clone karo
if data_path is None:
    repo_dir = Path("/content/flyrank-ml-internship-starter")

    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/flyrank-bih/flyrank-ml-internship-starter.git",
                str(repo_dir)
            ],
            check=True
        )

    data_path = repo_dir / "data/raw/content_refresh_anonymized.csv"

# Dataset load karo
df = pd.read_csv(data_path)

# Starter proxy label
declining_label = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Lane-relevant groups
visible = df["impressions_90d"] >= 500
stale = df["days_since_last_update"] >= 180
stale_visible = stale & visible

# Transparent hand-written baseline
hand_rule_score = (
    stale.astype(int)
    * visible.astype(int)
    * df["impressions_90d"]
)

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)
    ranking = np.argsort(-scores)
    return labels[ranking[:k]].mean()

hand_rule_p50 = precision_at_k(
    hand_rule_score,
    declining_label,
    50
)

print("Starter Dataset Evidence")
print("-" * 35)
print(f"Total pages: {len(df):,}")
print(f"Declining pages: {declining_label.sum():,}")
print(f"Declining rate: {declining_label.mean():.3f}")
print(f"Pages with at least 500 impressions: {visible.sum():,}")
print(f"Stale and visible pages: {stale_visible.sum():,}")
print(f"Hand-rule Precision@50: {hand_rule_p50:.3f}")

Starter Dataset Evidence
-----------------------------------
Total pages: 30,000
Declining pages: 16,262
Declining rate: 0.542
Pages with at least 500 impressions: 16,726
Stale and visible pages: 17
Hand-rule Precision@50: 0.680


## 4. Careful words: what I can and can't claim

## Careful Claims

### What this project may be able to say

This project may identify observable search, content, freshness, and engagement signals that are associated with pages receiving the declining proxy label.

It may also show whether a learned ranking places more declining pages near the top of a review queue than a transparent baseline rule.

The final output will be a decision-support tool. It may help a content or SEO team decide which pages deserve human review first.

### What this project cannot claim

This project cannot prove that any feature is a Google ranking factor.

It cannot claim that refreshing a recommended page will cause traffic or rankings to recover.

It cannot guarantee that every highly ranked page should be edited.

It cannot claim that results from the 30,000-row starter dataset will automatically generalize to the full warehouse release.

It also cannot treat `trend_direction` as a perfect final target because it describes the current measurement window rather than a future outcome.

The recommendations should therefore be described as observed, directional, and suitable for human review—not as automatic or causal decisions.

In [4]:
safe_claim = (
    "Observed page signals may help prioritize pages for human review."
)

claims_not_supported = [
    "The model proves Google's ranking algorithm.",
    "Refreshing a recommended page will definitely recover traffic.",
    "Every high-scoring page must be changed."
]

print("Safe claim:")
print(safe_claim)

print("\nClaims this project will not make:")
for claim in claims_not_supported:
    print("-", claim)

Safe claim:
Observed page signals may help prioritize pages for human review.

Claims this project will not make:
- The model proves Google's ranking algorithm.
- Refreshing a recommended page will definitely recover traffic.
- Every high-scoring page must be changed.


## Self-check

B## Completed Self-Check

- [x] I selected **Lane 2: Refresh / Content Opportunity Scoring**.
- [x] I explained why this lane is useful.
- [x] I defined the unit of analysis as one content page.
- [x] I named the decision being improved.
- [x] I described the human action after receiving a recommendation.
- [x] I explained the cost of a false positive.
- [x] I explained the cost of a false negative.
- [x] I showed at least two real numbers from the starter dataset.
- [x] I included a transparent baseline result.
- [x] I explained why this is not only a model-training exercise.
- [x] I used careful, observational language.
- [x] I did not include client names, URLs, domains, or private queries.
- [x] The notebook runs from top to bottom without errors.